<a href="https://colab.research.google.com/github/Rhefanza/datmin-maude/blob/main/notebooks/02_prapemrosesan_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tahap 2: Prapemrosesan dan Split Data
**UAS Data Mining II · MAUDE MRI (LNH) · event_type 4 kelas**

Notebook ini mengikuti Panduan Kerja MAUDE tahap 2 (langkah 2.1 sampai 2.6) secara berurutan.
Masukan: hasil tahap 1 (`data_bersih.csv`, 3.604 baris, sudah ada kolom `grup`).
Keluaran: `data/processed/data_eksperimen.csv`.



## 0. Persiapan

In [1]:
!pip install -U scikit-learn==1.8.0

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ----------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sklearn
print(sklearn.__version__)

1.8.0


In [3]:
# (Opsional) ambil repo supaya folder data/processed ada di dalam repo.
# Hapus tanda # di dua baris berikut kalau mau dipakai:
# !git clone https://github.com/Rhefanza/datmin-maude.git
# %cd datmin-maude

import re, os
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_colwidth", 120)

In [4]:
# Muat hasil tahap 1.
# Pilih SALAH SATU: (a) file sudah ada di folder kerja / Drive, atau (b) upload manual.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # jalankan dari akar repo agar outputs/ dan data/ konsisten
PATH_INPUT = "data_bersih.csv"   # ubah kalau lokasinya beda
for calon in ["data/interim/data_bersih.csv", "/kaggle/working/data/interim/data_bersih.csv"]:
    if not os.path.exists(PATH_INPUT) and os.path.exists(calon):
        PATH_INPUT = calon  # keluaran notebook 01

if not os.path.exists(PATH_INPUT):
    try:
        from google.colab import files
        print("File tidak ditemukan. Silakan upload data_bersih.csv:")
        up = files.upload()
        PATH_INPUT = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{PATH_INPUT} tidak ditemukan")

df = pd.read_csv(PATH_INPUT, keep_default_na=False,
                 dtype={"mdr_report_key": str, "date_received": str})
df["grup"] = df["grup"].astype(int)

# Cek masukan dari tahap 1
assert len(df) == 3604, f"Jumlah baris {len(df)}, seharusnya 3604"
assert df["norm"].is_unique, "Ada norm kembar"
assert (df["event_type"] != "").all(), "Ada label kosong"
print(df.shape)
print(df["event_type"].value_counts())

(3604, 7)
event_type
Injury         2144
Malfunction    1082
Other           342
Death            36
Name: count, dtype: int64


## 2.1 Daftar kata pembocor
Hapus: `injur*`, `serious*`, `harm*`, `malfunction*`, `death`, `deaths`.
Pertahankan: `died`, `expired`, `passed away`, `burn`.

In [5]:
# Batas kata memakai "bukan huruf a-z", supaya "injury2" atau "(injury)" tetap tertangkap
# tetapi "died", "expired", "burn" tidak ikut terhapus.
LEAK = re.compile(r"(?<![a-z])(?:(?:injur|serious|harm|malfunction)[a-z]*|deaths?(?![a-z]))")

# Tes cepat
for s in ["serious injury", "Injuries", "malfunctioning", "harmful", "death", "deaths",
          "died", "expired", "passed away", "burn", "no injury."]:
    print(f"{s!r:20} -> {LEAK.sub(' ', s.lower()).strip()!r}")

'serious injury'     -> ''
'Injuries'           -> ''
'malfunctioning'     -> ''
'harmful'            -> ''
'death'              -> ''
'deaths'             -> ''
'died'               -> 'died'
'expired'            -> 'expired'
'passed away'        -> 'passed away'
'burn'               -> 'burn'
'no injury.'         -> 'no  .'


In [6]:
# Tabel bukti: persen laporan yang memuat frasa tertentu, per kelas (pakai kolom norm)
frasa = ["serious injury", "injur", "malfunction", "harm", "death"]
bukti = pd.DataFrame({
    f: df["norm"].str.contains(f, regex=False).groupby(df["event_type"]).mean().mul(100).round(1)
    for f in frasa
}).T
bukti.index.name = "frasa (% laporan per kelas)"
os.makedirs("outputs/tables", exist_ok=True)
bukti.to_csv("outputs/tables/02_bukti_kata_pembocor.csv")
bukti   # acuan: "serious injury" 48,5% di Malfunction vs 5,4% di Injury

event_type,Death,Injury,Malfunction,Other
frasa (% laporan per kelas),,,,
serious injury,0.0,5.4,48.5,0.6
injur,13.9,24.9,65.5,15.8
malfunction,13.9,2.3,3.2,2.9
harm,0.0,1.5,4.4,1.8
death,33.3,0.8,1.2,0.0


## 2.2 Dua versi teks dan ganti nama label
- `text_clean` (model klasik): huruf kecil, buang semua selain a-z, hapus kata pembocor, rapikan spasi.
- `text_tf` (transformer): huruf kecil, hapus kata pembocor, tanda baca dan angka tetap ada.

In [7]:
def buat_text_clean(t):
    t = t.lower()
    t = re.sub(r"[^a-z]+", " ", t)       # buang semua selain a-z
    t = LEAK.sub(" ", t)                 # hapus kata pembocor
    return re.sub(r"\s+", " ", t).strip()

def buat_text_tf(t):
    t = t.lower()
    t = LEAK.sub(" ", t)                 # tanda baca dan angka tetap ada
    return re.sub(r"\s+", " ", t).strip()

df["text_clean"] = df["event_text"].map(buat_text_clean)
df["text_tf"]    = df["event_text"].map(buat_text_tf)
df = df.rename(columns={"event_type": "label"})

df[["event_text", "text_clean", "text_tf"]].head(3)

,event_text,text_clean,text_tf
0,OUTPATIENT UNDER GENERAL ANESTHESIA FOR LUMBAR SCAN INMRI. PATIENT RECEIVED BURNS UNDER 3 ECG ELECTRODES. BURNS NO...,outpatient under general anesthesia for lumbar scan inmri patient received burns under ecg electrodes burns noticed ...,outpatient under general anesthesia for lumbar scan inmri. patient received burns under 3 ecg electrodes. burns noti...
1,"PATIENT WAS BEING SCANNED IN THE THORACIC SPINE AREA USING THE QUAD T&L COIL. PATIENT ORIENTATION WAS HEAD FIRST, S...",patient was being scanned in the thoracic spine area using the quad t l coil patient orientation was head first supl...,"patient was being scanned in the thoracic spine area using the quad t&l coil. patient orientation was head first, su..."
2,PATIENT WAS UNDERGOING MRI SCANNING OF LT. KNEE FOR PAIN. PATIENT HAD RT. KNEE REPLACEMENT DONE ON 7/14/92. DURING...,patient was undergoing mri scanning of lt knee for pain patient had rt knee replacement done on during scan patient ...,patient was undergoing mri scanning of lt. knee for pain. patient had rt. knee replacement done on 7/14/92. during s...


## 2.3 Kolom tahun

In [8]:
df["tahun"] = df["date_received"].str[:4].astype(int)
print(df["tahun"].min(), "sampai", df["tahun"].max())

1992 sampai 2026


## 2.4 Split utama: 5 fold per grup
`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`, `y = label`, `groups = grup`.
Alasan 5-fold dan bukan 80/20: Death hanya 36, jadi satu split terlalu rapuh.

In [9]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1
for nomor, (_, idx_uji) in enumerate(sgkf.split(df, y=df["label"], groups=df["grup"])):
    df.loc[df.index[idx_uji], "fold"] = nomor

assert (df["fold"] >= 0).all()
pd.crosstab(df["fold"], df["label"], margins=True)   # acuan: tiap fold ~7 Death, ~68 Other

label,Death,Injury,Malfunction,Other,All
fold,,,,,
0,7,429,217,68,721
1,7,429,216,69,721
2,8,429,216,68,721
3,7,428,217,69,721
4,7,429,216,68,720
All,36,2144,1082,342,3604


## 2.5 Split temporal (pelengkap)
Latih: tahun <= 2023. Uji: 2024 sampai 2026.

In [10]:
latih = df[df["tahun"] <= 2023]
uji   = df[df["tahun"] >= 2024]
temporal = pd.DataFrame({"latih (<=2023)": latih["label"].value_counts(),
                         "uji (2024-2026)": uji["label"].value_counts()}).fillna(0).astype(int)
temporal.loc["Total"] = temporal.sum()
temporal.to_csv("outputs/tables/02_split_temporal.csv")
temporal

,latih (<=2023),uji (2024-2026)
label,,
Death,32,4
Injury,1768,376
Malfunction,514,568
Other,330,12
Total,2644,960


In [11]:
# Grup kemiripan tidak boleh terbelah antara bagian latih dan uji temporal
sisi = df.assign(uji=df["tahun"] >= 2024).groupby("grup")["uji"].nunique()
assert (sisi == 1).all(), f"{(sisi > 1).sum()} grup menyeberang batas 2023/2024"
print("Grup yang menyeberang split temporal:", int((sisi > 1).sum()))


Grup yang menyeberang split temporal: 0


In [12]:
# Keterbatasan yang perlu ditulis di laporan:
print("Death di bagian uji :", (uji["label"] == "Death").sum())
print("Other di bagian uji :", (uji["label"] == "Other").sum())
# Kalau angkanya sangat kecil, tulis sebagai keterbatasan validasi temporal.

Death di bagian uji : 4
Other di bagian uji : 12


## Cek akhir tahap 2
1. Tabel silang fold x label: tiap fold punya sekitar 7 Death dan 68 Other.
2. Tidak ada grup yang muncul di lebih dari satu fold.
3. Pencarian kata pembocor di `text_clean` dan `text_tf` menghasilkan 0.

In [13]:
# 1. fold x label sudah ditampilkan di 2.4
cek1 = df.groupby("fold")["label"].value_counts().unstack()
print("Death per fold:", cek1["Death"].tolist(), "| Other per fold:", cek1["Other"].tolist())

# 2. tidak ada grup di lebih dari satu fold
maks_fold_per_grup = df.groupby("grup")["fold"].nunique().max()
assert maks_fold_per_grup == 1, "Ada grup yang tersebar di >1 fold!"
print("Maks fold per grup:", maks_fold_per_grup, "(harus 1)")

# 3. tidak ada kata pembocor tersisa
sisa_clean = df["text_clean"].map(lambda x: bool(LEAK.search(x))).sum()
sisa_tf    = df["text_tf"].map(lambda x: bool(LEAK.search(x))).sum()
assert sisa_clean == 0 and sisa_tf == 0
print("Kata pembocor tersisa -> text_clean:", sisa_clean, "| text_tf:", sisa_tf, "(harus 0)")

# Tambahan: tidak ada teks yang jadi kosong setelah dibersihkan
print("text_clean kosong:", (df["text_clean"] == "").sum(), "| text_tf kosong:", (df["text_tf"] == "").sum())

Death per fold: [7, 7, 8, 7, 7] | Other per fold: [68, 69, 68, 69, 68]
Maks fold per grup: 1 (harus 1)
Kata pembocor tersisa -> text_clean: 0 | text_tf: 0 (harus 0)
text_clean kosong: 0 | text_tf kosong: 0


## 2.6 Bekukan dan bagikan


In [14]:
from datetime import date

KOLOM = ["mdr_report_key", "date_received", "tahun", "manufacturer", "label",
         "grup", "fold", "event_text", "norm", "text_clean", "text_tf"]
os.makedirs("data/processed", exist_ok=True)
df[KOLOM].to_csv("data/processed/data_eksperimen.csv", index=False)

cek = pd.read_csv("data/processed/data_eksperimen.csv", keep_default_na=False,
                  dtype={"mdr_report_key": str, "date_received": str})
print("Baris tersimpan :", len(cek))
print("Kolom           :", list(cek.columns))
print("Tanggal bekukan :", date.today().isoformat())
print("\nKabari tim: data_eksperimen.csv beku,", len(cek), "baris, tanggal", date.today().isoformat())

Baris tersimpan : 3604
Kolom           : ['mdr_report_key', 'date_received', 'tahun', 'manufacturer', 'label', 'grup', 'fold', 'event_text', 'norm', 'text_clean', 'text_tf']
Tanggal bekukan : 2026-11-23

Kabari tim: data_eksperimen.csv beku, 3604 baris, tanggal 2026-11-23


In [15]:
# Unduh file hasil (kalau jalan di Colab) supaya bisa di-upload ke GitHub / dibagikan ke tim
try:
    from google.colab import files
    files.download("data/processed/data_eksperimen.csv")
except ImportError:
    pass